# 02.05 — Pipeline ETL completo con PySpark

## Objetivo

En los notebooks anteriores estudiamos individualmente los principales componentes de PySpark:

- modelo de ejecución;
- DataFrames;
- transformaciones;
- agregaciones;
- joins;
- Window Functions;
- lectura y escritura;
- Parquet;
- particionado.

En este notebook integraremos estos conceptos dentro de un pequeño **pipeline ETL de extremo a extremo**.

El objetivo será transformar datos transaccionales sin procesar en datasets preparados para análisis.

El flujo será:

```text
RAW
 │
 │ datos originales
 ▼
TRANSFORMED
 │
 │ limpieza + validaciones + enriquecimiento
 ▼
CURATED
 │
 │ agregaciones y métricas de negocio
 ▼
PARQUET PARTICIONADO
```

## 1. Contexto del problema

Supongamos que una empresa almacena información de:

```text
clientes
+
transacciones
```

Los datos originales contienen algunos problemas habituales:

- valores faltantes;
- registros duplicados;
- transacciones inválidas;
- clientes inexistentes;
- fechas almacenadas como texto;
- diferentes tipos de cliente.

Nuestro objetivo será construir dos datasets finales.

### Dataset transaccional enriquecido

Una fila por transacción:

```text
id_transaccion
id_cliente
nombre
estado
tipo_cliente
fecha
categoria
monto
anio
mes
numero_compra
gasto_acumulado
dias_desde_compra_anterior
```

### Dataset agregado por cliente

Una fila por cliente:

```text
id_cliente
nombre
estado
tipo_cliente
num_transacciones
gasto_total
ticket_promedio
compra_maxima
primera_compra
ultima_compra
dias_entre_primera_ultima
```

## 2. Preparar el entorno

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DoubleType
)

spark = (
    SparkSession.builder
    .appName("02_05_spark_etl_pipeline")
    .getOrCreate()
)

print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(
    "Default parallelism:",
    spark.sparkContext.defaultParallelism
)

Spark version: 4.1.3
Master: local[*]
Default parallelism: 12


## 3. Definir rutas

Utilizaremos una estructura sencilla para simular las diferentes capas del pipeline.

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw" / "02_05_etl"
TRANSFORMED_DIR = DATA_DIR / "transformed" / "02_05_etl"
CURATED_DIR = DATA_DIR / "curated" / "02_05_etl"

for directory in [
    RAW_DIR,
    TRANSFORMED_DIR,
    CURATED_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\derec\Desktop\Proyectos\BigData_project


Nuestro pipeline utilizará:

```text
data/
├── raw/
│   └── 02_05_etl/
│
├── transformed/
│   └── 02_05_etl/
│
└── curated/
    └── 02_05_etl/
```


## 4. Capa RAW

La capa **Raw** representa los datos originales tal como llegan desde la fuente.

En esta etapa buscamos conservar la información con la menor cantidad posible de modificaciones.

### 4.1 Clientes

In [3]:
clientes_data = [
    (1, "Ana",     "CDMX",       "Premium", "ana@email.com"),
    (2, "Luis",    "Jalisco",    "Básico",  "luis@email.com"),
    (3, "Marta",   "Nuevo León", "Premium", None),
    (4, "Carlos",  "CDMX",       "Básico",  "carlos@email.com"),
    (5, "Sofía",   "Puebla",     "Básico",  "sofia@email.com"),
    (6, "Jorge",   "Jalisco",    "Premium", "jorge@email.com"),
    (7, "Elena",   "CDMX",       "Premium", None),
    (8, "Diego",   "Querétaro",  "Básico",  "diego@email.com"),
    (9, "Laura",   "Puebla",     "Premium", "laura@email.com"),
    (10, "Pedro",  "CDMX",       "Premium", "pedro@email.com"),
    (11, "Andrea", "Jalisco",    "Básico",  "andrea@email.com"),
    (12, "Miguel", None,         "Básico",  "miguel@email.com"),

    # Duplicado intencional
    (12, "Miguel", None, "Básico", "miguel@email.com")
]

In [4]:
clientes_schema = StructType([
    StructField("id_cliente", IntegerType(), False),
    StructField("nombre", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("tipo_cliente", StringType(), True),
    StructField("email", StringType(), True)
])

In [5]:
df_clientes_raw = spark.createDataFrame(
    clientes_data,
    schema=clientes_schema
)

df_clientes_raw.show(
    truncate=False
)

+----------+------+----------+------------+----------------+
|id_cliente|nombre|estado    |tipo_cliente|email           |
+----------+------+----------+------------+----------------+
|1         |Ana   |CDMX      |Premium     |ana@email.com   |
|2         |Luis  |Jalisco   |Básico      |luis@email.com  |
|3         |Marta |Nuevo León|Premium     |NULL            |
|4         |Carlos|CDMX      |Básico      |carlos@email.com|
|5         |Sofía |Puebla    |Básico      |sofia@email.com |
|6         |Jorge |Jalisco   |Premium     |jorge@email.com |
|7         |Elena |CDMX      |Premium     |NULL            |
|8         |Diego |Querétaro |Básico      |diego@email.com |
|9         |Laura |Puebla    |Premium     |laura@email.com |
|10        |Pedro |CDMX      |Premium     |pedro@email.com |
|11        |Andrea|Jalisco   |Básico      |andrea@email.com|
|12        |Miguel|NULL      |Básico      |miguel@email.com|
|12        |Miguel|NULL      |Básico      |miguel@email.com|
+----------+------+-----

### 4.2 Transacciones

In [6]:
transacciones_data = [
    (1001, 1,  "2026-01-05", 1200.0, "Electrónica"),
    (1002, 1,  "2026-02-18",  850.0, "Hogar"),
    (1003, 1,  "2026-05-09", 2200.0, "Electrónica"),

    (1004, 2,  "2026-01-22",  450.0, "Ropa"),
    (1005, 2,  "2026-04-11",  780.0, "Hogar"),

    (1006, 3,  "2026-02-03", 3200.0, "Electrónica"),
    (1007, 3,  "2026-03-17", 1800.0, "Ropa"),
    (1008, 3,  "2026-06-21", 4100.0, "Electrónica"),

    (1009, 4,  "2026-03-04", 350.0, "Hogar"),

    (1010, 5,  "2026-01-30", 650.0, "Ropa"),
    (1011, 5,  "2026-05-16", 900.0, "Ropa"),

    (1012, 6,  "2026-02-27", 2700.0, "Electrónica"),
    (1013, 6,  "2026-07-01", 1950.0, "Hogar"),

    (1014, 7,  "2026-04-08", 1450.0, "Electrónica"),
    (1015, 7,  "2026-06-12", 2300.0, "Electrónica"),

    (1016, 8,  "2026-03-26", 520.0, "Ropa"),

    (1017, 9,  "2026-02-14", 1300.0, "Hogar"),
    (1018, 9,  "2026-05-28", 1700.0, "Electrónica"),

    (1019, 10, "2026-01-12", 2900.0, "Electrónica"),
    (1020, 10, "2026-03-20", 1100.0, "Hogar"),
    (1021, 10, "2026-07-15", 3500.0, "Electrónica"),

    # Cliente inexistente
    (1022, 99, "2026-06-30", 5000.0, "Electrónica"),

    # Monto inválido
    (1023, 2, "2026-07-20", -300.0, "Ropa"),

    # Duplicado intencional
    (1021, 10, "2026-07-15", 3500.0, "Electrónica")
]

In [7]:
transacciones_schema = StructType([
    StructField("id_transaccion", IntegerType(), False),
    StructField("id_cliente", IntegerType(), False),
    StructField("fecha", StringType(), True),
    StructField("monto", DoubleType(), True),
    StructField("categoria", StringType(), True)
])

In [8]:
df_transacciones_raw = spark.createDataFrame(
    transacciones_data,
    schema=transacciones_schema
)

df_transacciones_raw.show(
    truncate=False
)

+--------------+----------+----------+------+-----------+
|id_transaccion|id_cliente|fecha     |monto |categoria  |
+--------------+----------+----------+------+-----------+
|1001          |1         |2026-01-05|1200.0|Electrónica|
|1002          |1         |2026-02-18|850.0 |Hogar      |
|1003          |1         |2026-05-09|2200.0|Electrónica|
|1004          |2         |2026-01-22|450.0 |Ropa       |
|1005          |2         |2026-04-11|780.0 |Hogar      |
|1006          |3         |2026-02-03|3200.0|Electrónica|
|1007          |3         |2026-03-17|1800.0|Ropa       |
|1008          |3         |2026-06-21|4100.0|Electrónica|
|1009          |4         |2026-03-04|350.0 |Hogar      |
|1010          |5         |2026-01-30|650.0 |Ropa       |
|1011          |5         |2026-05-16|900.0 |Ropa       |
|1012          |6         |2026-02-27|2700.0|Electrónica|
|1013          |6         |2026-07-01|1950.0|Hogar      |
|1014          |7         |2026-04-08|1450.0|Electrónica|
|1015         

## 5. Escribir la capa Raw

Guardaremos ambos datasets sin transformaciones analíticas.

In [9]:
CLIENTES_RAW_PATH = str(
    RAW_DIR / "clientes"
)

TRANSACCIONES_RAW_PATH = str(
    RAW_DIR / "transacciones"
)

In [10]:
(
    df_clientes_raw
    .write
    .mode("overwrite")
    .parquet(CLIENTES_RAW_PATH)
)

(
    df_transacciones_raw
    .write
    .mode("overwrite")
    .parquet(TRANSACCIONES_RAW_PATH)
)

Con esto simulamos el primer paso de ingestión:

```text
fuente
  ↓
RAW
```


## 6. Leer nuevamente desde Raw

A partir de este momento trabajaremos sobre los datos almacenados.

In [11]:
df_clientes = (
    spark.read
    .parquet(CLIENTES_RAW_PATH)
)

df_transacciones = (
    spark.read
    .parquet(TRANSACCIONES_RAW_PATH)
)

In [12]:
print(
    "Clientes Raw:",
    df_clientes.count()
)

print(
    "Transacciones Raw:",
    df_transacciones.count()
)

Clientes Raw: 13
Transacciones Raw: 24


## 7. Controles iniciales de calidad

Antes de transformar los datos revisaremos algunos problemas básicos.

### 7.1 Duplicados de clientes

In [13]:
(
    df_clientes
    .groupBy("id_cliente")
    .count()
    .filter(
        F.col("count") > 1
    )
    .show()
)

+----------+-----+
|id_cliente|count|
+----------+-----+
|        12|    2|
+----------+-----+



### 7.2 Duplicados de transacciones

In [14]:
(
    df_transacciones
    .groupBy("id_transaccion")
    .count()
    .filter(
        F.col("count") > 1
    )
    .show()
)

+--------------+-----+
|id_transaccion|count|
+--------------+-----+
|          1021|    2|
+--------------+-----+



### 7.3 Transacciones con monto inválido

In [15]:
df_transacciones.filter(
    F.col("monto") <= 0
).show()

+--------------+----------+----------+------+---------+
|id_transaccion|id_cliente|     fecha| monto|categoria|
+--------------+----------+----------+------+---------+
|          1023|         2|2026-07-20|-300.0|     Ropa|
+--------------+----------+----------+------+---------+



### 7.4 Valores nulos

In [16]:
df_clientes.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in df_clientes.columns
    ]
).show()

+----------+------+------+------------+-----+
|id_cliente|nombre|estado|tipo_cliente|email|
+----------+------+------+------------+-----+
|         0|     0|     2|           0|    2|
+----------+------+------+------------+-----+



## 8. Integridad referencial

Queremos comprobar si todas las transacciones corresponden a un cliente existente.

Para ello utilizamos `left_anti`.

In [17]:
df_transacciones_huerfanas = (
    df_transacciones
    .join(
        df_clientes.select(
            "id_cliente"
        ),
        on="id_cliente",
        how="left_anti"
    )
)

df_transacciones_huerfanas.show(
    truncate=False
)

+----------+--------------+----------+------+-----------+
|id_cliente|id_transaccion|fecha     |monto |categoria  |
+----------+--------------+----------+------+-----------+
|99        |1022          |2026-06-30|5000.0|Electrónica|
+----------+--------------+----------+------+-----------+



Deberíamos detectar:

```text
id_cliente = 99
```

Esta validación representa un control de:

```text
integridad referencial
```

## 9. Capa Transformed — clientes

Ahora comenzamos la limpieza.

Queremos:

- eliminar duplicados;
- estandarizar nombres;
- completar estados faltantes;
- completar tipos faltantes;
- crear indicador de email.


In [18]:
df_clientes_clean = (
    df_clientes

    .dropDuplicates(
        ["id_cliente"]
    )

    .withColumn(
        "nombre",
        F.initcap(
            F.trim("nombre")
        )
    )

    .fillna({
        "estado": "Sin información",
        "tipo_cliente": "Sin clasificar"
    })

    .withColumn(
        "email_disponible",
        F.when(
            F.col("email").isNotNull(),
            True
        ).otherwise(False)
    )
)

In [19]:
df_clientes_clean.show(
    truncate=False
)

+----------+------+---------------+------------+----------------+----------------+
|id_cliente|nombre|estado         |tipo_cliente|email           |email_disponible|
+----------+------+---------------+------------+----------------+----------------+
|1         |Ana   |CDMX           |Premium     |ana@email.com   |true            |
|2         |Luis  |Jalisco        |Básico      |luis@email.com  |true            |
|3         |Marta |Nuevo León     |Premium     |NULL            |false           |
|4         |Carlos|CDMX           |Básico      |carlos@email.com|true            |
|5         |Sofía |Puebla         |Básico      |sofia@email.com |true            |
|6         |Jorge |Jalisco        |Premium     |jorge@email.com |true            |
|7         |Elena |CDMX           |Premium     |NULL            |false           |
|8         |Diego |Querétaro      |Básico      |diego@email.com |true            |
|9         |Laura |Puebla         |Premium     |laura@email.com |true            |
|10 

## 10. Capa Transformed — transacciones

Aplicaremos las siguientes reglas:

1. eliminar duplicados;
2. eliminar montos no positivos;
3. convertir fecha a tipo `date`;
4. eliminar registros sin fecha válida;
5. conservar únicamente clientes existentes;
6. agregar año y mes.

In [20]:
df_transacciones_clean = (
    df_transacciones

    .dropDuplicates(
        ["id_transaccion"]
    )

    .filter(
        F.col("monto") > 0
    )

    .withColumn(
        "fecha",
        F.to_date(
            "fecha",
            "yyyy-MM-dd"
        )
    )

    .filter(
        F.col("fecha").isNotNull()
    )

    .join(
        df_clientes_clean.select(
            "id_cliente"
        ),
        on="id_cliente",
        how="left_semi"
    )

    .withColumn(
        "anio",
        F.year("fecha")
    )

    .withColumn(
        "mes",
        F.month("fecha")
    )
)

In [21]:
df_transacciones_clean.show(
    truncate=False
)

+----------+--------------+----------+------+-----------+----+---+
|id_cliente|id_transaccion|fecha     |monto |categoria  |anio|mes|
+----------+--------------+----------+------+-----------+----+---+
|1         |1001          |2026-01-05|1200.0|Electrónica|2026|1  |
|1         |1002          |2026-02-18|850.0 |Hogar      |2026|2  |
|1         |1003          |2026-05-09|2200.0|Electrónica|2026|5  |
|2         |1004          |2026-01-22|450.0 |Ropa       |2026|1  |
|2         |1005          |2026-04-11|780.0 |Hogar      |2026|4  |
|3         |1006          |2026-02-03|3200.0|Electrónica|2026|2  |
|3         |1007          |2026-03-17|1800.0|Ropa       |2026|3  |
|3         |1008          |2026-06-21|4100.0|Electrónica|2026|6  |
|4         |1009          |2026-03-04|350.0 |Hogar      |2026|3  |
|5         |1010          |2026-01-30|650.0 |Ropa       |2026|1  |
|5         |1011          |2026-05-16|900.0 |Ropa       |2026|5  |
|6         |1012          |2026-02-27|2700.0|Electrónica|2026|

## 11. Validar la limpieza

Comparamos registros antes y después.

In [22]:
print(
    "Transacciones originales:",
    df_transacciones.count()
)

print(
    "Transacciones limpias:",
    df_transacciones_clean.count()
)

Transacciones originales: 24
Transacciones limpias: 21


También verificamos que no queden montos inválidos.

In [23]:
print(
    "Montos inválidos:",
    df_transacciones_clean
    .filter(
        F.col("monto") <= 0
    )
    .count()
)

Montos inválidos: 0


Y que ya no existan transacciones huérfanas:

In [24]:
num_huerfanas = (
    df_transacciones_clean
    .join(
        df_clientes_clean.select(
            "id_cliente"
        ),
        on="id_cliente",
        how="left_anti"
    )
    .count()
)

print(
    "Transacciones huérfanas:",
    num_huerfanas
)

Transacciones huérfanas: 0


## 12. Escribir capa Transformed

Guardaremos los datos ya limpios.

In [25]:
CLIENTES_TRANSFORMED_PATH = str(
    TRANSFORMED_DIR /
    "clientes"
)

TRANSACCIONES_TRANSFORMED_PATH = str(
    TRANSFORMED_DIR /
    "transacciones"
)

In [26]:
(
    df_clientes_clean
    .write
    .mode("overwrite")
    .parquet(
        CLIENTES_TRANSFORMED_PATH
    )
)

Para transacciones utilizaremos particionado físico por año:

In [27]:
(
    df_transacciones_clean
    .write
    .mode("overwrite")
    .partitionBy("anio")
    .parquet(
        TRANSACCIONES_TRANSFORMED_PATH
    )
)

Nuestro pipeline ya tiene:

```text
RAW
 ↓
limpieza
 ↓
TRANSFORMED
```

## 13. Construir dataset transaccional enriquecido

Ahora combinaremos clientes y transacciones.

In [28]:
df_transacciones_enriched = (
    df_transacciones_clean
    .join(
        df_clientes_clean,
        on="id_cliente",
        how="left"
    )
)

Este dataset ya contiene:

```text
transacción
+
información del cliente
```


## 14. Agregar Window Functions

Queremos calcular por cliente:

- número secuencial de compra;
- gasto acumulado;
- monto anterior;
- fecha anterior;
- días desde la compra anterior.

Definimos las ventanas.

In [29]:
w_cliente_fecha = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        "fecha",
        "id_transaccion"
    )
)

In [30]:
w_acumulada = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        "fecha",
        "id_transaccion"
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

## 15. Generar métricas transaccionales

In [31]:
df_transacciones_enriched = (
    df_transacciones_enriched

    .withColumn(
        "numero_compra",
        F.row_number().over(
            w_cliente_fecha
        )
    )

    .withColumn(
        "gasto_acumulado",
        F.sum("monto").over(
            w_acumulada
        )
    )

    .withColumn(
        "monto_anterior",
        F.lag(
            "monto",
            1
        ).over(
            w_cliente_fecha
        )
    )

    .withColumn(
        "fecha_anterior",
        F.lag(
            "fecha",
            1
        ).over(
            w_cliente_fecha
        )
    )

    .withColumn(
        "dias_desde_compra_anterior",
        F.datediff(
            F.col("fecha"),
            F.col("fecha_anterior")
        )
    )
)

## 16. Seleccionar columnas finales

In [32]:
df_transacciones_curated = (
    df_transacciones_enriched

    .select(
        "id_transaccion",
        "id_cliente",
        "nombre",
        "estado",
        "tipo_cliente",
        "fecha",
        "anio",
        "mes",
        "categoria",
        "monto",
        "numero_compra",
        "gasto_acumulado",
        "monto_anterior",
        "dias_desde_compra_anterior"
    )

    .orderBy(
        "id_cliente",
        "fecha",
        "id_transaccion"
    )
)

In [33]:
df_transacciones_curated.show(
    truncate=False
)

+--------------+----------+------+----------+------------+----------+----+---+-----------+------+-------------+---------------+--------------+--------------------------+
|id_transaccion|id_cliente|nombre|estado    |tipo_cliente|fecha     |anio|mes|categoria  |monto |numero_compra|gasto_acumulado|monto_anterior|dias_desde_compra_anterior|
+--------------+----------+------+----------+------------+----------+----+---+-----------+------+-------------+---------------+--------------+--------------------------+
|1001          |1         |Ana   |CDMX      |Premium     |2026-01-05|2026|1  |Electrónica|1200.0|1            |1200.0         |NULL          |NULL                      |
|1002          |1         |Ana   |CDMX      |Premium     |2026-02-18|2026|2  |Hogar      |850.0 |2            |2050.0         |1200.0        |44                        |
|1003          |1         |Ana   |CDMX      |Premium     |2026-05-09|2026|5  |Electrónica|2200.0|3            |4250.0         |850.0         |80      

## 17. Construir resumen por cliente

Ahora crearemos nuestro segundo dataset analítico.

Partimos de las transacciones limpias.

In [34]:
df_metricas_cliente = (
    df_transacciones_clean

    .groupBy(
        "id_cliente"
    )

    .agg(
        F.count(
            "id_transaccion"
        ).alias(
            "num_transacciones"
        ),

        F.round(
            F.sum("monto"),
            2
        ).alias(
            "gasto_total"
        ),

        F.round(
            F.avg("monto"),
            2
        ).alias(
            "ticket_promedio"
        ),

        F.max(
            "monto"
        ).alias(
            "compra_maxima"
        ),

        F.min(
            "fecha"
        ).alias(
            "primera_compra"
        ),

        F.max(
            "fecha"
        ).alias(
            "ultima_compra"
        ),

        F.countDistinct(
            "categoria"
        ).alias(
            "categorias_distintas"
        )
    )
)

## 18. Incluir clientes sin transacciones

Queremos que nuestro dataset final contenga **todos los clientes**, incluso aquellos que no hayan realizado compras.

Para ello:

In [35]:
df_clientes_curated = (
    df_clientes_clean

    .join(
        df_metricas_cliente,
        on="id_cliente",
        how="left"
    )

    .fillna({
        "num_transacciones": 0,
        "gasto_total": 0.0,
        "ticket_promedio": 0.0,
        "compra_maxima": 0.0,
        "categorias_distintas": 0
    })

    .withColumn(
        "dias_entre_primera_ultima",
        F.datediff(
            F.col("ultima_compra"),
            F.col("primera_compra")
        )
    )
)

## 19. Crear segmento de comportamiento

Podemos agregar una variable derivada.

In [36]:
df_clientes_curated = (
    df_clientes_curated

    .withColumn(
        "segmento_valor",

        F.when(
            F.col("gasto_total") == 0,
            "Sin actividad"
        )

        .when(
            F.col("gasto_total") < 2000,
            "Bajo"
        )

        .when(
            F.col("gasto_total") < 5000,
            "Medio"
        )

        .otherwise(
            "Alto"
        )
    )
)

In [37]:
df_clientes_curated = (
    df_clientes_curated

    .select(
        "id_cliente",
        "nombre",
        "estado",
        "tipo_cliente",
        "email_disponible",
        "num_transacciones",
        "gasto_total",
        "ticket_promedio",
        "compra_maxima",
        "categorias_distintas",
        "primera_compra",
        "ultima_compra",
        "dias_entre_primera_ultima",
        "segmento_valor"
    )

    .orderBy(
        F.col(
            "gasto_total"
        ).desc()
    )
)

In [38]:
df_clientes_curated.show(
    truncate=False
)

+----------+------+---------------+------------+----------------+-----------------+-----------+---------------+-------------+--------------------+--------------+-------------+-------------------------+--------------+
|id_cliente|nombre|estado         |tipo_cliente|email_disponible|num_transacciones|gasto_total|ticket_promedio|compra_maxima|categorias_distintas|primera_compra|ultima_compra|dias_entre_primera_ultima|segmento_valor|
+----------+------+---------------+------------+----------------+-----------------+-----------+---------------+-------------+--------------------+--------------+-------------+-------------------------+--------------+
|3         |Marta |Nuevo León     |Premium     |false           |3                |9100.0     |3033.33        |4100.0       |2                   |2026-02-03    |2026-06-21   |138                      |Alto          |
|10        |Pedro |CDMX           |Premium     |true            |3                |7500.0     |2500.0         |3500.0       |2      

## 21. Validaciones del dataset Curated

Un pipeline no debería terminar simplemente porque:

```text
el código corrió
```

También debemos comprobar que el resultado tiene sentido.


### 21.1 Un registro por cliente

In [39]:
total_clientes = (
    df_clientes_curated
    .count()
)

clientes_unicos = (
    df_clientes_curated
    .select(
        "id_cliente"
    )
    .distinct()
    .count()
)

print(
    "Registros:",
    total_clientes
)

print(
    "Clientes únicos:",
    clientes_unicos
)

Registros: 12
Clientes únicos: 12


Esperamos:

```text
Registros = Clientes únicos
```

### 21.2 Gasto no negativo

In [40]:
gastos_invalidos = (
    df_clientes_curated
    .filter(
        F.col("gasto_total") < 0
    )
    .count()
)

print(
    "Gastos inválidos:",
    gastos_invalidos
)

Gastos inválidos: 0


Esperamos:

```text
0
```

### 21.3 Número de transacciones válido

In [41]:
transacciones_invalidas = (
    df_clientes_curated
    .filter(
        F.col("num_transacciones") < 0
    )
    .count()
)

print(
    "Conteos inválidos:",
    transacciones_invalidas
)

Conteos inválidos: 0


Esperamos:

```text
0
```

## 22. Reconciliación de métricas

Podemos comprobar que el gasto total de las transacciones coincide con el gasto agregado por cliente.

### Total desde transacciones

In [42]:
total_transacciones = (
    df_transacciones_clean
    .agg(
        F.sum("monto")
        .alias("total")
    )
    .first()["total"]
)

### Total desde clientes

In [43]:
total_clientes = (
    df_clientes_curated
    .agg(
        F.sum("gasto_total")
        .alias("total")
    )
    .first()["total"]
)

In [44]:
print(
    "Total transacciones:",
    total_transacciones
)

print(
    "Total clientes:",
    total_clientes
)

print(
    "Diferencia:",
    total_transacciones
    - total_clientes
)

Total transacciones: 35900.0
Total clientes: 35900.0
Diferencia: 0.0


Esperamos una diferencia:

```text
0
```

o prácticamente cero por cuestiones de precisión numérica.

Esta clase de validación se conoce como:

```text
reconciliación
```

y es muy importante en pipelines reales.

## 23. Escribir capa Curated

Ahora escribiremos los dos datasets finales.

In [45]:
TRANSACTIONS_CURATED_PATH = str(
    CURATED_DIR /
    "transacciones_enriquecidas"
)

CLIENTES_CURATED_PATH = str(
    CURATED_DIR /
    "clientes_resumen"
)

### Dataset transaccional

Como las consultas podrían realizarse frecuentemente por año y mes:

In [46]:
(
    df_transacciones_curated

    .write
    .mode("overwrite")

    .partitionBy(
        "anio",
        "mes"
    )

    .parquet(
        TRANSACTIONS_CURATED_PATH
    )
)

### Dataset por cliente

In [47]:
(
    df_clientes_curated

    .write
    .mode("overwrite")

    .parquet(
        CLIENTES_CURATED_PATH
    )
)

Nuestro pipeline completo queda:

```text
                   FUENTES
                      │
                      ▼
                     RAW
                      │
         ┌────────────┴────────────┐
         │                         │
     clientes                transacciones
         │                         │
         └────────────┬────────────┘
                      │
                 limpieza
                      │
            controles de calidad
                      │
                      ▼
                 TRANSFORMED
                      │
               join + windows
               + agregaciones
                      │
                      ▼
                   CURATED
                 ┌────┴────┐
                 │         │
          transacciones   clientes
           enriquecidas   resumen
```

## 24. Comprobar Partition Pruning

Leemos únicamente marzo de 2026.

In [48]:
df_marzo = (
    spark.read
    .parquet(
        TRANSACTIONS_CURATED_PATH
    )

    .filter(
        (F.col("anio") == 2026)
        &
        (F.col("mes") == 3)
    )
)

In [49]:
df_marzo.show(
    truncate=False
)

+--------------+----------+------+----------+------------+----------+---------+------+-------------+---------------+--------------+--------------------------+----+---+
|id_transaccion|id_cliente|nombre|estado    |tipo_cliente|fecha     |categoria|monto |numero_compra|gasto_acumulado|monto_anterior|dias_desde_compra_anterior|anio|mes|
+--------------+----------+------+----------+------------+----------+---------+------+-------------+---------------+--------------+--------------------------+----+---+
|1007          |3         |Marta |Nuevo León|Premium     |2026-03-17|Ropa     |1800.0|2            |5000.0         |3200.0        |42                        |2026|3  |
|1009          |4         |Carlos|CDMX      |Básico      |2026-03-04|Hogar    |350.0 |1            |350.0          |NULL          |NULL                      |2026|3  |
|1016          |8         |Diego |Querétaro |Básico      |2026-03-26|Ropa     |520.0 |1            |520.0          |NULL          |NULL                      |20

Inspeccionamos:

In [50]:
df_marzo.explain(
    "formatted"
)

== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [14]: [id_transaccion#2214, id_cliente#2215, nombre#2216, estado#2217, tipo_cliente#2218, fecha#2219, categoria#2220, monto#2221, numero_compra#2222, gasto_acumulado#2223, monto_anterior#2224, dias_desde_compra_anterior#2225, anio#2226, mes#2227]
Batched: true
Location: InMemoryFileIndex [file:/c:/Users/derec/Desktop/Proyectos/BigData_project/data/curated/02_05_etl/transacciones_enriquecidas]
PartitionFilters: [isnotnull(anio#2226), isnotnull(mes#2227), (anio#2226 = 2026), (mes#2227 = 3)]
ReadSchema: struct<id_transaccion:int,id_cliente:int,nombre:string,estado:string,tipo_cliente:string,fecha:date,categoria:string,monto:double,numero_compra:int,gasto_acumulado:double,monto_anterior:double,dias_desde_compra_anterior:int>

(2) ColumnarToRow [codegen id : 1]
Input [14]: [id_transaccion#2214, id_cliente#2215, nombre#2216, estado#2217, tipo_cliente#2218, fecha#2219, categoria#2220, monto#2221, numero_co

Busca:

```text
PartitionFilters
```

y condiciones relacionadas con:

```text
anio = 2026
mes = 3
```

Esto conecta nuestro pipeline final con lo aprendido en `02_04`.

## 25. Inspeccionar el plan del pipeline

Podemos observar cómo Spark ejecutará la construcción del dataset transaccional.

In [51]:
df_transacciones_curated.explain(
    "formatted"
)

== Physical Plan ==
AdaptiveSparkPlan (33)
+- Sort (32)
   +- Exchange (31)
      +- Project (30)
         +- Window (29)
            +- Sort (28)
               +- Exchange (27)
                  +- Project (26)
                     +- BroadcastHashJoin LeftOuter BuildRight (25)
                        :- Project (16)
                        :  +- BroadcastHashJoin LeftSemi BuildRight (15)
                        :     :- Project (8)
                        :     :  +- Filter (7)
                        :     :     +- SortAggregate (6)
                        :     :        +- Sort (5)
                        :     :           +- Exchange (4)
                        :     :              +- SortAggregate (3)
                        :     :                 +- Sort (2)
                        :     :                    +- Scan parquet  (1)
                        :     +- BroadcastExchange (14)
                        :        +- HashAggregate (13)
                        :           +- 

Intenta localizar:

```text
Exchange
Sort
Window
Join
Project
Filter
```

y relacionarlos con nuestro código:

```text
Filter
    ↓
limpieza

Join
    ↓
clientes + transacciones

Exchange / Sort
    ↓
preparación Window

Window
    ↓
métricas temporales

Project
    ↓
selección final
```

## 26. Identificar operaciones costosas

Después de los cinco notebooks ya podemos hacer una primera lectura crítica del pipeline.

Tenemos operaciones potencialmente costosas como:

```text
join
groupBy
Window
orderBy
partitionBy al escribir
```

porque pueden implicar:

```text
shuffle
+
sort
+
movimiento de datos
```

Mientras que operaciones como:

```text
filter
select
withColumn
```

suelen ser transformaciones más locales.

Todavía no intentaremos optimizar agresivamente el pipeline.

Eso formará parte de una fase posterior.

La meta aquí es ser capaces de **identificar dónde podrían aparecer los costos**.

## 27. Pipeline ETL como función

Hasta ahora escribimos el proceso paso por paso para comprenderlo.

En un proyecto real es conveniente separar las transformaciones en funciones.

Por ejemplo:

In [52]:
def clean_clients(df):
    return (
        df
        .dropDuplicates(
            ["id_cliente"]
        )
        .withColumn(
            "nombre",
            F.initcap(
                F.trim("nombre")
            )
        )
        .fillna({
            "estado": "Sin información",
            "tipo_cliente": "Sin clasificar"
        })
        .withColumn(
            "email_disponible",
            F.col("email").isNotNull()
        )
    )

In [53]:
def clean_transactions(
    df_transactions,
    df_clients
):
    return (
        df_transactions

        .dropDuplicates(
            ["id_transaccion"]
        )

        .filter(
            F.col("monto") > 0
        )

        .withColumn(
            "fecha",
            F.to_date(
                "fecha",
                "yyyy-MM-dd"
            )
        )

        .filter(
            F.col("fecha").isNotNull()
        )

        .join(
            df_clients.select(
                "id_cliente"
            ),
            on="id_cliente",
            how="left_semi"
        )

        .withColumn(
            "anio",
            F.year("fecha")
        )

        .withColumn(
            "mes",
            F.month("fecha")
        )
    )

Nuestro flujo podría entonces parecerse a:

In [54]:
clients_clean = clean_clients(
    df_clientes
)

transactions_clean = clean_transactions(
    df_transacciones,
    clients_clean
)

Esto permite separar:

```text
lógica
de
orquestación
```

y será útil cuando posteriormente movamos código fuera de los notebooks hacia:

```text
src/
```

## 28. Principios utilizados en el pipeline

Durante este notebook aplicamos varios principios importantes.

### 1. Separación por capas

```text
Raw
↓
Transformed
↓
Curated
```

Cada capa tiene una responsabilidad diferente.

### 2. Schema conocido

Intentamos evitar depender innecesariamente de inferencia de tipos.

### 3. Calidad de datos

Validamos:

```text
duplicados
nulos
montos inválidos
integridad referencial
```

### 4. Transformaciones reproducibles

El pipeline produce el mismo resultado a partir de los mismos datos.

### 5. Formato analítico

Utilizamos:

```text
Parquet
```

como formato principal.

### 6. Particionado físico

Organizamos el dataset transaccional mediante:

```text
anio
mes
```

### 7. Reconciliación

Comprobamos que las métricas agregadas coincidan con los datos fuente transformados.


## 29. Resumen del pipeline

Podemos representar todo el trabajo realizado en este notebook de la siguiente manera:

```text
                 INGESTA
                    │
                    ▼
                   RAW
                    │
                    ▼
          ┌─────────────────┐
          │ Quality Checks  │
          └─────────────────┘
                    │
                    ▼
                LIMPIEZA
                    │
          ┌─────────┴─────────┐
          │                   │
      clientes           transacciones
      limpios               limpias
          │                   │
          └─────────┬─────────┘
                    │
                   JOIN
                    │
                    ▼
               ENRIQUECIMIENTO
                    │
              Window Functions
                    │
                    ▼
             TRANSACCIONES
               CURATED
                    │
                    │
            Aggregations
                    │
                    ▼
                CLIENTES
                 CURATED
                    │
                    ▼
              PARQUET
                    │
              partitionBy
                    │
                    ▼
          DATASETS ANALÍTICOS
```

## 30. Conclusiones

Este notebook integra los fundamentos estudiados durante toda la fase de PySpark.

Utilizamos:

```text
DataFrames
Schemas
filter()
select()
withColumn()
when()
joins
left_semi
left_anti
groupBy()
agg()
Window
row_number()
lag()
sum().over()
Parquet
partitionBy()
explain()
```

pero la principal diferencia respecto a los notebooks anteriores es que aquí no utilizamos estas herramientas de forma aislada.

Las combinamos dentro de un proceso completo:

```text
ingesta
  ↓
calidad
  ↓
limpieza
  ↓
transformación
  ↓
enriquecimiento
  ↓
agregación
  ↓
validación
  ↓
persistencia
```

Este flujo representa la base de un pipeline ETL con Spark.

A partir de aquí el siguiente nivel ya no consiste principalmente en aprender nuevas funciones de DataFrames, sino en trabajar sobre problemas como:

- mayor volumen de datos;
- optimización;
- skew;
- shuffles;
- estrategias de join;
- manejo de archivos;
- configuración de recursos;
- ejecución distribuida;
- arquitectura de pipelines.